# Project Overview
This repository contains an exploratory data analysis (EDA) and operational case study of Instacart consumer purchasing patterns, traffic flows, and reorder dynamics. Leveraging a local PostgreSQL database, Python (pandas, sqlalchemy), and interactive Plotly visualizations, this project uncovers the behavioral drivers behind grocery delivery logistics—highlighting key metrics like the fixed ~10-item basket invariant and the 100K product volume reorder stability threshold.

In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
import plotly.express as px
from sqlalchemy import create_engine

load_dotenv()

db_user = os.getenv('DB_USER')
db_password = os.getenv('DB_PASSWORD')
db_host = os.getenv('DB_HOST')
db_port = os.getenv('DB_PORT')
db_name = os.getenv('DB_NAME')

engine = create_engine(
    f'postgresql+psycopg://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}'
)

print('Database engine created successfully!')


# Query was created to determine the average basket size across each day of the week, and to get a count of orders and of how many products are 
# sold each day of the week. 

dow_query = """
    SELECT 
        o.order_dow,
        COUNT(DISTINCT o.order_id) AS total_orders,
        COUNT(op.product_id) AS total_items_sold,
        ROUND(CAST(COUNT(op.product_id) AS NUMERIC) / COUNT(DISTINCT o.order_id), 2) AS avg_basket_size
    FROM order_products op
    JOIN orders o ON op.order_id = o.order_id
    GROUP BY o.order_dow
    ORDER BY o.order_dow;
"""
df_dow = pd.read_sql(dow_query, engine)

# Maps the integer days to readable labels
dow_map = {0: 'Sun', 1: 'Mon', 2: 'Tue', 3: 'Wed', 4: 'Thu', 5: 'Fri', 6: 'Sat'}
df_dow['day_name'] = df_dow['order_dow'].map(dow_map)

# Creates a bar graph of the total count of orders by each day of the week, due to limitations of the dataset.
fig_dow = px.bar(
    df_dow, 
    x='day_name', 
    y='total_orders',
    title='<b>Instacart Order Volume by Day of the Week</b>',
    labels={'day_name': 'Day of Week', 'total_orders': 'Total Orders'},
    template='plotly_white',
    color='total_orders',
    color_continuous_scale='Blues'
)
fig_dow.update_layout(showlegend=False, title_font_size=16)
fig_dow.show()


# Query to determine the average reorder rate for products based on department.
dept_query = """
    SELECT 
        d.department,
        COUNT(op.product_id) AS total_purchased,
        ROUND(CAST(SUM(op.reordered) AS NUMERIC) / COUNT(op.product_id) * 100, 2) AS reorder_rate_pct
    FROM order_products op
    JOIN products p ON op.product_id = p.product_id
    JOIN departments d ON p.department_id = d.department_id
    GROUP BY d.department
    ORDER BY reorder_rate_pct DESC;
"""
df_dept = pd.read_sql(dept_query, engine)

# Creates horizontal bar graph from highest to lowest reorder rate by department.
fig_dept = px.bar(
    df_dept, 
    x='reorder_rate_pct', 
    y='department',
    orientation='h',
    title='<b>Department Reorder Rates</b>',
    labels={'reorder_rate_pct': 'Reorder Rate (%)', 'department': 'Department'},
    template='plotly_white'
)
fig_dept.update_layout(yaxis={'categoryorder':'total ascending'}, title_font_size=16)
fig_dept.show()


# Query for the top 100 products by order count, and calculates their percentage reorder rate.
product_query = """
    SELECT 
        p.product_name,
        d.department,
        COUNT(op.product_id) AS total_orders,
        ROUND(CAST(SUM(op.reordered) AS NUMERIC) / COUNT(op.product_id) * 100, 2) AS reorder_rate_pct
    FROM order_products op
    JOIN products p ON op.product_id = p.product_id
    JOIN departments d ON p.department_id = d.department_id
    GROUP BY p.product_name, d.department
    HAVING COUNT(op.product_id) > 500
    ORDER BY total_orders DESC
    LIMIT 100;
"""
df_products = pd.read_sql(product_query, engine)

# Graphs a scatterplot of the top 100 products by order count along the x-axis and their reorder rate along the y-axis. Color is based on department.
fig_scatter = px.scatter(
    df_products, 
    x='total_orders', 
    y='reorder_rate_pct',
    color='department',
    hover_name='product_name',
    title='<b>Product Volume vs. Reorder Rate</b>',
    labels={'total_orders': 'Total Order Volume', 'reorder_rate_pct': 'Reorder Rate (%)'},
    template='plotly_white'
)
fig_scatter.update_layout(title_font_size=16)
fig_scatter.show()


# Instacart Volume by Day of the Week
Sunday and Monday have the largest amount of orders with about 58K vs about 44K for each of the rest of the days. 
Surprisingly the average basket size doesn't have any significant variance from day to day with each being around 10 items total.

# Department Reorder Rates
Based on department, reorder rate looks to be most likely just due to shelflife or how quickly an item gets used up, with dairy eggs, beverages, and produce all being at the top for reorder rate. These products are generally gone through quickly or spoil after a short duration.

# Product Volume vs. Reorder Rate
Splits up the department items further into their specific products and gives count of their total order volume and specific reorder rates.  Overall there is a trend for items with higher Total order volume count having a higher reorder rate but the trend isn't fully consistent for the lower volume count side. Around to 35K to 100K items varies greatly on reorder rate unlike the higher count items. Shelf life, usage rate (consumption) of product, and order volume all correlate to a higher reorder rate but only for a total volume above 100K does the noise start to disappear.

In [ ]:
# Query for count of orders by hour of the day.
hourly_query = """
    SELECT 
        order_hour_of_day,
        COUNT(DISTINCT order_id) AS total_orders
    FROM orders
    GROUP BY order_hour_of_day
    ORDER BY order_hour_of_day;
"""
df_hourly = pd.read_sql(hourly_query, engine)

# Line graph of the total volume of orders for each hour of the day.
fig_hourly = px.line(
    df_hourly, 
    x='order_hour_of_day', 
    y='total_orders',
    markers=True,
    title='<b>Instacart Order Volume by Hour of the Day</b>',
    labels={'order_hour_of_day': 'Hour of Day (0–23)', 'total_orders': 'Total Orders'},
    template='plotly_white'
)
fig_hourly.update_layout(
    xaxis=dict(tickmode='linear', dtick=1),
    title_font_size=16
)
fig_hourly.show()

# Query for the count of orders by the hour of the day for each day of the week.
hourly_dow_query = """
    SELECT 
        o.order_dow,
        o.order_hour_of_day,
        COUNT(DISTINCT o.order_id) AS total_orders
    FROM orders o
    GROUP BY o.order_dow, o.order_hour_of_day
    ORDER BY o.order_dow, o.order_hour_of_day;
"""
df_hourly_dow = pd.read_sql(hourly_dow_query, engine)

# Converts the integers to their meaning, each day of the week.
df_hourly_dow['day_name'] = df_hourly_dow['order_dow'].map({
    0: 'Sun', 1: 'Mon', 2: 'Tue', 3: 'Wed', 4: 'Thu', 5: 'Fri', 6: 'Sat'
})

# Chronologically orders each day of the week in the legend instead of alphabetical.
day_order = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']
df_hourly_dow['day_name'] = pd.Categorical(df_hourly_dow['day_name'], categories=day_order, ordered=True)
df_hourly_dow = df_hourly_dow.sort_values(['day_name', 'order_hour_of_day'])

# Multiple Line Graph, line for each day of the weeks total volume of orders for each hour of the day.
fig_hourly_dow = px.line(
    df_hourly_dow, 
    x='order_hour_of_day', 
    y='total_orders',
    color='day_name',
    markers=False,  
    title='<b>Hourly Order Volume by Day of the Week</b>',
    labels={
        'order_hour_of_day': 'Hour of Day (0–23)', 
        'total_orders': 'Total Orders',
        'day_name': 'Day of Week'
    },
    template='plotly_white'
)

fig_hourly_dow.update_layout(
    xaxis=dict(tickmode='linear', dtick=1),
    title_font_size=16,
    legend_title_text='Day'
)

fig_hourly_dow.show()

# 

# Instacart Order Volume by Hour of the Day
Graphs the total number of orders for each hour of the day. Orders happen at every hour of the day but are extremely low from 1AM to 5am. Orders start to climb steadily at 6am to a plateau around 9am until 4pm and decline steadily to a low valley at 1am.

# Hourly Order Volume by Day of the Week
Graphs the days of the week each as a separate line for its total orders for each hour of the day.  They all follow the same trends as previous graph just with higher peaks for Sunday and Monday.

In [ ]:
# Query for the count of orders for each specific day of the week.
box_query = """
    SELECT 
        o.order_id,
        o.order_dow,
        COUNT(op.product_id) AS items_per_order
    FROM order_products op
    JOIN orders o ON op.order_id = o.order_id
    GROUP BY o.order_id, o.order_dow;
"""
df_box = pd.read_sql(box_query, engine)

# Change the integers to their corresponding day of the week
df_box['day_name'] = df_box['order_dow'].map({
    0: 'Sun', 1: 'Mon', 2: 'Tue', 3: 'Wed', 4: 'Thu', 5: 'Fri', 6: 'Sat'
})

# Sets the days to chronological order instead of alphabetical.
day_order = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']
df_box['day_name'] = pd.Categorical(df_box['day_name'], categories=day_order, ordered=True)
df_box = df_box.sort_values('day_name')

# Creates a box and whisker plot of the for the basket size for each day of the week. Set range to 35 since outliers stretch too far (to 145).
fig_box = px.box(
    df_box, 
    x='day_name', 
    y='items_per_order',
    title='<b>Basket Size Distribution by Day of the Week</b>',
    labels={'day_name': 'Day of Week', 'items_per_order': 'Number of Items per Order'},
    template='plotly_white',
    color='day_name'
)

fig_box.update_layout(
    yaxis=dict(range=[0, 35]), 
    showlegend=False,
    title_font_size=16
)

fig_box.show()

# Basket Size Distribution by Day of the Week
There is very little variance for each day of the week with the largest variance being the max for each days with Tuesday having a max for 144 and Sunday a max of 99. Overall I would just attribute that to randomness as Sunday has the most overall orders and even a slightly higher median than all but Saturday which is equal to it.